In [5]:
import pandas as pd

from tradepy.models import *

In [20]:
df = pd.read_csv('../Data/gc1_final.csv')

In [21]:
df["datetime"] = pd.to_datetime(df["datetime"])

# df["ticker"] = df["ticker"].astype("category")
# df["ticker_5min"] = df["ticker_5min"].astype("category")

df = df.sort_values("datetime").reset_index(drop=True)

df = df.drop(columns=["date", "per", "per_5min", 'ticker', 'ticker_5min'])

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 925349 entries, 0 to 925348
Columns: 115 entries, datetime to close_daily
dtypes: bool(1), datetime64[us](1), float64(105), int64(8)
memory usage: 805.7 MB


In [30]:
linear_regression_model(df.drop(columns=["datetime","close_5min","close_daily"]).dropna())

Mean Squared Error: 0.03242659236177867
R^2 Score: 0.9999995319227918


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [33]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, roc_auc_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans, DBSCAN

In [38]:
def linear_regression_model_try(df):
    df = df.copy()

    # 1. Crear target ANTES del dropna
    df["target"] = df["close"].shift(-5) / df["close"] - 1

    # 2. Eliminar filas con NaN (incluye las últimas 5)
    df = df.dropna()

    # 3. Separar X e Y DESPUÉS del dropna
    X = df.drop(columns=["close", "target", "datetime", "close_5min", "close_daily"])
    Y = df["target"]

    # 4. Split temporal (NO aleatorio)
    train_size = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
    Y_train, Y_test = Y.iloc[:train_size], Y.iloc[train_size:]

    # 5. Entrenar
    model = LinearRegression()
    model.fit(X_train, Y_train)

    # 6. Evaluar
    Y_pred = model.predict(X_test)
    mse = mean_squared_error(Y_test, Y_pred)
    r2 = r2_score(Y_test, Y_pred)

    print(f"Mean Squared Error: {mse}")
    print(f"R^2 Score: {r2}")
    return model


In [39]:
linear_regression_model_try(df)

Mean Squared Error: 1.4853572285264296e-06
R^2 Score: -0.0024217401318051834


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [40]:
def lstm_model_try(df, horizon=5, window_size=60):
    df = df.copy()

    # ============================
    # 1. Crear target futuro
    # ============================
    df["target"] = df["close"].shift(-horizon) / df["close"] - 1

    # ============================
    # 2. Eliminar NaNs (últimas filas)
    # ============================
    df = df.dropna().reset_index(drop=True)

    # ============================
    # 3. Seleccionar features
    # ============================
    feature_cols = [c for c in df.columns if c not in ["datetime", "target"]]
    X = df[feature_cols].values
    Y = df["target"].values

    # ============================
    # 4. Escalado multivariante
    # ============================
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # ============================
    # 5. Crear ventanas temporales
    # ============================
    X_seq, Y_seq = [], []
    for i in range(window_size, len(X_scaled)):
        X_seq.append(X_scaled[i-window_size:i, :])  # TODAS las features
        Y_seq.append(Y[i])  # target futuro

    X_seq = np.array(X_seq)
    Y_seq = np.array(Y_seq)

    # ============================
    # 6. Split temporal (80/20)
    # ============================
    train_size = int(len(X_seq) * 0.8)
    X_train, X_test = X_seq[:train_size], X_seq[train_size:]
    Y_train, Y_test = Y_seq[:train_size], Y_seq[train_size:]

    # ============================
    # 7. Modelo LSTM
    # ============================
    model = Sequential()
    model.add(LSTM(64, return_sequences=True, input_shape=(window_size, X_seq.shape[2])))
    model.add(Dropout(0.2))
    model.add(LSTM(32, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(1))

    model.compile(optimizer="adam", loss="mse")

    # ============================
    # 8. Entrenamiento
    # ============================
    model.fit(X_train, Y_train, epochs=20, batch_size=32, verbose=1)

    # ============================
    # 9. Predicción
    # ============================
    Y_pred = model.predict(X_test)

    # ============================
    # 10. Métricas
    # ============================
    mse = mean_squared_error(Y_test, Y_pred)
    mae = mean_absolute_error(Y_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_test, Y_pred)

    print(f"MSE: {mse}")
    print(f"MAE: {mae}")
    print(f"MAPE: {mape}")

    # ============================
    # 11. Plot
    # ============================
    plt.figure(figsize=(12, 6))
    plt.plot(Y_test, label="Actual Future Return")
    plt.plot(Y_pred, label="Predicted Future Return", alpha=0.7)
    plt.legend()
    plt.title("LSTM Future Return Prediction")
    plt.show()

    return model


In [41]:
lstm_model_try(df)

MemoryError: Unable to allocate 46.2 GiB for an array with shape (906801, 60, 114) and data type float64

In [42]:
import tensorflow as tf


class WindowGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, Y, window_size, batch_size=64):
        self.X = X
        self.Y = Y
        self.window_size = window_size
        self.batch_size = batch_size

    def __len__(self):
        return (len(self.X) - self.window_size) // self.batch_size

    def __getitem__(self, idx):
        X_batch = []
        Y_batch = []

        start = idx * self.batch_size
        end = start + self.batch_size

        for i in range(start, end):
            X_batch.append(self.X[i:i+self.window_size])
            Y_batch.append(self.Y[i+self.window_size])

        return np.array(X_batch), np.array(Y_batch)


def lstm_model_try_2(df, horizon=5, window_size=60):
    df = df.copy()

    # Target futuro
    df["target"] = df["close"].shift(-horizon) / df["close"] - 1
    df = df.dropna().reset_index(drop=True)

    # Features
    feature_cols = [c for c in df.columns if c not in ["datetime", "target"]]
    X = df[feature_cols].values
    Y = df["target"].values

    # Escalado
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # Split temporal
    split = int(len(X_scaled) * 0.8)
    X_train, X_test = X_scaled[:split], X_scaled[split:]
    Y_train, Y_test = Y[:split], Y[split:]

    # Generadores
    train_gen = WindowGenerator(X_train, Y_train, window_size)
    test_gen = WindowGenerator(X_test, Y_test, window_size)

    # Modelo LSTM
    model = tf.keras.Sequential([
        tf.keras.layers.LSTM(64, return_sequences=True, input_shape=(window_size, X.shape[1])),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1)
    ])

    model.compile(optimizer="adam", loss="mse")

    # Entrenamiento
    model.fit(train_gen, epochs=10, validation_data=test_gen)

    # Predicción
    Y_pred = model.predict(test_gen)

    # Métricas
    mse = mean_squared_error(Y_test[window_size:], Y_pred)
    print("MSE:", mse)

    return model


In [43]:
lstm_model_try_2(df)

c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


c:\Users\LIJUN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 325s 28ms/step - loss: 2.0619e-04 - val_loss: 1.5086e-06
Epoch 2/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 280s 25ms/step - loss: 2.0222e-06 - val_loss: 2.2010e-06
Epoch 3/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 278s 25ms/step - loss: 1.9194e-06 - val_loss: 1.5170e-06
Epoch 4/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 317s 28ms/step - loss: 1.9180e-06 - val_loss: 1.6196e-06
Epoch 5/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 281s 25ms/step - loss: 1.9264e-06 - val_loss: 1.6588e-06
Epoch 6/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 280s 25ms/step - loss: 1.9179e-06 - val_loss: 1.9185e-06
Epoch 7/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 282s 25ms/step - loss: 1.9270e-06 - val_loss: 2.2237e-06
Epoch 8/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 282s 25ms/step - loss: 1.9161e-06 - val_loss: 1.5378e-06
Epoch 9/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 280s 25ms/step - loss: 1.9160e-06 - val_loss: 1.4905e-06
Epoch 10/10
11334/11334 ━━━━━━━━━━━━━━━━━━━━ 280s 25ms/step - loss: 1.9271e-06 - val_loss: 

ValueError: Found input variables with inconsistent numbers of samples: [181313, 181312]